In [ ]:
import torch
import numpy as np
import pandas as pd
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from scipy.stats import entropy
from scipy.special import kl_div
import warnings
warnings.filterwarnings('ignore')

# Load model and tokenizer
print("Loading GPT-2...")
model_name = "gpt2"
tokenizer = GPT2TokenizerFast.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)
model.eval()
tokenizer.pad_token = tokenizer.eos_token
print("Model loaded successfully.")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading GPT-2...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model loaded successfully.
Number of parameters: 124,439,808


In [ ]:
print(model)
print("Done")


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)
Done


In [ ]:
print(type(model))
print("Model loaded:", model is not None)

<class 'transformers.models.gpt2.modeling_gpt2.GPT2LMHeadModel'>
Model loaded: True


In [ ]:
def get_token_probabilities(text, context=None, max_length=256):
    """
    Get token-level probability distributions from GPT-2.
    If context is provided, prepend it to the text (RAG scenario).
    If not, just use the text alone (no-retrieval scenario).
    """
    if context:
        input_text = f"Context: {context[:500]}\n\nAnswer: {text}"
    else:
        input_text = text

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=max_length,
        truncation=True
    )

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits  # shape: [1, seq_len, vocab_size]

    # Convert logits to probabilities
    probs = torch.softmax(logits, dim=-1)  # [1, seq_len, vocab_size]
    probs = probs[0].numpy()  # [seq_len, vocab_size]

    return probs

def compute_entropy(probs):
    """
    Compute Shannon entropy for each token position.
    High entropy = model is uncertain.
    Low entropy = model is confident.
    """
    entropies = []
    for token_probs in probs:
        h = entropy(token_probs)  # Shannon entropy
        entropies.append(h)
    return np.array(entropies)

# Quick test
sample_text = "The capital of France is Paris."
sample_context = "France is a country in Western Europe. Its capital city is Paris."

probs_no_context = get_token_probabilities(sample_text)
probs_with_context = get_token_probabilities(sample_text, context=sample_context)

entropy_no_ctx = compute_entropy(probs_no_context)
entropy_with_ctx = compute_entropy(probs_with_context)

print(f"Tokens without context: {len(entropy_no_ctx)}")
print(f"Tokens with context: {len(entropy_with_ctx)}")
print(f"Mean entropy WITHOUT context: {entropy_no_ctx.mean():.4f}")
print(f"Mean entropy WITH context: {entropy_with_ctx.mean():.4f}")
print(f"Entropy dropped by: {entropy_no_ctx.mean() - entropy_with_ctx.mean():.4f}")
print("\nCore functions working correctly!")

Tokens without context: 7
Tokens with context: 27
Mean entropy WITHOUT context: 4.9111
Mean entropy WITH context: 3.7514
Entropy dropped by: 1.1597

Core functions working correctly!


In [ ]:
def compute_all_metrics(text, context, max_length=256):
    """
    Compute all 5 Track A metrics for a single sample.
    Returns a dictionary of scores.
    """
    # Get probability distributions with and without context
    probs_no_ctx = get_token_probabilities(text, context=None, max_length=max_length)
    probs_with_ctx = get_token_probabilities(text, context=context, max_length=max_length)

    # Align lengths (take minimum to compare token by token)
    min_len = min(len(probs_no_ctx), len(probs_with_ctx))
    probs_no_ctx = probs_no_ctx[:min_len]
    probs_with_ctx = probs_with_ctx[:min_len]

    # --- Metric 1: Entropy Only (Baseline 1) ---
    # Mean entropy of output without context
    entropy_no_ctx = compute_entropy(probs_no_ctx)
    entropy_with_ctx = compute_entropy(probs_with_ctx)
    entropy_only = float(np.mean(entropy_no_ctx))

    # --- Metric 2: Information Gain (IG) ---
    # How much does entropy DROP when context is added?
    # Higher IG = context helped = less likely hallucination
    ig_per_token = entropy_no_ctx - entropy_with_ctx
    information_gain = float(np.mean(ig_per_token))

    # --- Metric 3: KL Divergence ---
    # How different are the distributions with vs without context?
    # Higher KL = context changed the model a lot
    kl_per_token = []
    for p_no, p_with in zip(probs_no_ctx, probs_with_ctx):
        # Add small epsilon to avoid log(0)
        p_no = p_no + 1e-10
        p_with = p_with + 1e-10
        p_no = p_no / p_no.sum()
        p_with = p_with / p_with.sum()
        kl = float(np.sum(kl_div(p_no, p_with)))
        kl_per_token.append(kl)
    kl_divergence = float(np.mean(kl_per_token))

    # --- Metric 4: Confidence Drop ---
    # How much does the max token probability change with context?
    # Negative drop = context made model MORE confident (good sign)
    conf_no_ctx = np.max(probs_no_ctx, axis=1)
    conf_with_ctx = np.max(probs_with_ctx, axis=1)
    confidence_drop = float(np.mean(conf_no_ctx - conf_with_ctx))

    # --- Metric 5: Semantic Entropy ---
    # Entropy computed on top-10 tokens only (captures meaning-level uncertainty)
    sem_entropies = []
    for token_probs in probs_with_ctx:
        top10_idx = np.argsort(token_probs)[-10:]
        top10_probs = token_probs[top10_idx]
        top10_probs = top10_probs / top10_probs.sum()
        sem_h = entropy(top10_probs)
        sem_entropies.append(sem_h)
    semantic_entropy = float(np.mean(sem_entropies))

    return {
        'entropy_only': entropy_only,
        'information_gain': information_gain,
        'kl_divergence': kl_divergence,
        'confidence_drop': confidence_drop,
        'semantic_entropy': semantic_entropy,
        'ig_per_token': ig_per_token.tolist(),
        'entropy_per_token': entropy_with_ctx.tolist()
    }

# Test on sample
result = compute_all_metrics(sample_text, sample_context)
print("All metrics for sample:")
for k, v in result.items():
    if 'per_token' not in k:
        print(f"  {k}: {v:.4f}")

All metrics for sample:
  entropy_only: 4.9111
  information_gain: -0.6417
  kl_divergence: 4.3914
  confidence_drop: 0.0378
  semantic_entropy: 1.8678


In [ ]:
from sklearn.preprocessing import MinMaxScaler

def compute_composite_score(metrics):
    """
    Combine all metrics into one hallucination score.
    Higher score = more likely hallucinated.
    """
    # For hallucination detection:
    # - Low IG = context didn't help = hallucination signal
    # - High entropy = uncertain = hallucination signal  
    # - High KL = big distribution shift = hallucination signal
    # - High confidence drop = model less confident with context = hallucination signal
    # - High semantic entropy = meaning-level uncertainty = hallucination signal

    score = (
        -metrics['information_gain'] * 0.35 +   # flip sign: low IG = bad
         metrics['entropy_only']    * 0.25 +
         metrics['kl_divergence']   * 0.20 +
         metrics['confidence_drop'] * 0.10 +
         metrics['semantic_entropy']* 0.10
    )
    return score

# Run on a small batch of 50 samples to test pipeline
print("Running pipeline on 50 test samples...")
print("(This will take a few minutes on CPU)")

df_rag = pd.read_csv('data/ragtruth.csv')
test_batch = df_rag.sample(50, random_state=42).reset_index(drop=True)

results = []
for i, row in test_batch.iterrows():
    try:
        metrics = compute_all_metrics(
            str(row['output'])[:300],
            str(row['context'])[:500]
        )
        metrics['is_hallucinated'] = row['is_hallucinated']
        metrics['halu_type'] = row['halu_type']
        metrics['model'] = row['model']
        metrics['composite'] = compute_composite_score(metrics)
        results.append(metrics)
        if (len(results)) % 10 == 0:
            print(f"  Processed {len(results)}/50 samples...")
    except Exception as e:
        print(f"  Skipped sample {i}: {e}")

df_results = pd.DataFrame(results)
print(f"\nDone! Processed {len(df_results)} samples.")
print("\nMean composite score — hallucinated vs not:")
print(df_results.groupby('is_hallucinated')['composite'].mean())


Running pipeline on 50 test samples...
(This will take a few minutes on CPU)
  Processed 10/50 samples...
  Processed 20/50 samples...
  Processed 30/50 samples...
  Processed 40/50 samples...
  Processed 50/50 samples...

Done! Processed 50 samples.

Mean composite score — hallucinated vs not:
is_hallucinated
0    2.631715
1    2.749017
Name: composite, dtype: float64


In [ ]:
from sklearn.metrics import roc_auc_score, f1_score
from scipy.stats import spearmanr

labels = df_results['is_hallucinated'].values
composite_scores = df_results['composite'].values

# AUROC
auroc = roc_auc_score(labels, composite_scores)
print(f"AUROC on 50-sample test batch: {auroc:.4f}")

# Spearman correlation
rho, pval = spearmanr(composite_scores, labels)
print(f"Spearman rho: {rho:.4f} (p={pval:.4f})")

# Individual metric AUROCs
for metric in ['entropy_only', 'information_gain', 'kl_divergence', 
               'confidence_drop', 'semantic_entropy']:
    scores = df_results[metric].values
    try:
        if 'information_gain' in metric:
            auc = roc_auc_score(labels, -scores)  # flip sign
        else:
            auc = roc_auc_score(labels, scores)
        print(f"  AUROC {metric}: {auc:.4f}")
    except:
        print(f"  AUROC {metric}: could not compute")

AUROC on 50-sample test batch: 0.6763
Spearman rho: 0.3051 (p=0.0312)
  AUROC entropy_only: 0.5881
  AUROC information_gain: 0.3333
  AUROC kl_divergence: 0.6971
  AUROC confidence_drop: 0.3333
  AUROC semantic_entropy: 0.3462


In [ ]:
import time

print("Running pipeline on 500 samples...")
print("Estimated time: 20-30 minutes on CPU")
print("Do not close Jupyter during this process.\n")

# Use stratified sample to maintain hallucination ratio
df_halu_samples = df_rag[df_rag['is_hallucinated']==1].sample(250, random_state=42)
df_clean_samples = df_rag[df_rag['is_hallucinated']==0].sample(250, random_state=42)
df_sample = pd.concat([df_halu_samples, df_clean_samples]).reset_index(drop=True)

full_results = []
start_time = time.time()

for i, row in df_sample.iterrows():
    try:
        metrics = compute_all_metrics(
            str(row['output'])[:300],
            str(row['context'])[:500]
        )
        metrics['is_hallucinated'] = row['is_hallucinated']
        metrics['halu_type'] = row['halu_type']
        metrics['model'] = row['model']
        metrics['task_type'] = row['task_type']
        metrics['composite'] = compute_composite_score(metrics)
        full_results.append(metrics)

        if len(full_results) % 50 == 0:
            elapsed = time.time() - start_time
            rate = len(full_results) / elapsed
            remaining = (500 - len(full_results)) / rate
            print(f"  {len(full_results)}/500 done — "
                  f"~{remaining/60:.1f} minutes remaining")
    except Exception as e:
        continue

df_full = pd.DataFrame(full_results)
df_full.to_csv('results/ragtruth_results.csv', index=False)
print(f"\nDone! {len(df_full)} samples processed.")
print(f"Total time: {(time.time()-start_time)/60:.1f} minutes")
print("Results saved to results/ragtruth_results.csv")

Running pipeline on 500 samples...
Estimated time: 20-30 minutes on CPU
Do not close Jupyter during this process.

  50/500 done — ~14.6 minutes remaining
  100/500 done — ~12.7 minutes remaining
  150/500 done — ~11.1 minutes remaining
  200/500 done — ~10.1 minutes remaining
  250/500 done — ~8.8 minutes remaining
  300/500 done — ~7.1 minutes remaining
  350/500 done — ~5.3 minutes remaining
  400/500 done — ~3.6 minutes remaining
  450/500 done — ~1.8 minutes remaining
  500/500 done — ~0.0 minutes remaining

Done! 500 samples processed.
Total time: 18.5 minutes
Results saved to results/ragtruth_results.csv


In [ ]:
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.calibration import calibration_curve
from scipy.stats import spearmanr, bootstrap
import numpy as np

df_full = pd.read_csv('results/ragtruth_results.csv')
labels = df_full['is_hallucinated'].values

def compute_auroc_with_ci(scores, labels, n_bootstrap=1000):
    """Compute AUROC with 95% bootstrap confidence interval."""
    auroc = roc_auc_score(labels, scores)
    
    bootstrap_aurocs = []
    for _ in range(n_bootstrap):
        idx = np.random.choice(len(labels), len(labels), replace=True)
        try:
            b_auroc = roc_auc_score(labels[idx], scores[idx])
            bootstrap_aurocs.append(b_auroc)
        except:
            continue
    
    ci_lower = np.percentile(bootstrap_aurocs, 2.5)
    ci_upper = np.percentile(bootstrap_aurocs, 97.5)
    return auroc, ci_lower, ci_upper

def compute_ece(scores, labels, n_bins=10):
    """Compute Expected Calibration Error."""
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    # Normalize scores to [0,1]
    scores_norm = (scores - scores.min()) / (scores.max() - scores.min() + 1e-10)
    for i in range(n_bins):
        mask = (scores_norm >= bins[i]) & (scores_norm < bins[i+1])
        if mask.sum() == 0:
            continue
        bin_acc = labels[mask].mean()
        bin_conf = scores_norm[mask].mean()
        ece += mask.sum() * abs(bin_acc - bin_conf)
    return ece / len(labels)

def compute_f1(scores, labels):
    """Compute F1 at best threshold."""
    thresholds = np.percentile(scores, np.arange(10, 90, 5))
    best_f1 = 0
    for thresh in thresholds:
        preds = (scores >= thresh).astype(int)
        f1 = f1_score(labels, preds, zero_division=0)
        best_f1 = max(best_f1, f1)
    return best_f1

print("=" * 65)
print(f"{'Metric':<25} {'AUROC':>8} {'95% CI':>16} {'F1':>8} {'Spearman':>10} {'ECE':>8}")
print("=" * 65)

metrics_config = [
    ('Entropy only (B1)',    'entropy_only',     False),
    ('+ Information Gain',  'information_gain',  True),
    ('+ KL divergence',     'kl_divergence',     False),
    ('+ Confidence drop',   'confidence_drop',   False),
    ('+ Semantic entropy',  'semantic_entropy',  False),
    ('Full composite',      'composite',         False),
]

for label_name, col, flip in metrics_config:
    scores = df_full[col].values
    if flip:
        scores = -scores
    
    auroc, ci_low, ci_high = compute_auroc_with_ci(scores, labels)
    f1 = compute_f1(scores, labels)
    rho, _ = spearmanr(scores, labels)
    ece = compute_ece(scores, labels)
    
    print(f"{label_name:<25} {auroc:>8.4f} [{ci_low:.3f}-{ci_high:.3f}] {f1:>8.4f} {rho:>10.4f} {ece:>8.4f}")

print("=" * 65)
print("\nNote: SelfCheckGPT baseline will be added in 03_baselines.ipynb")

Metric                       AUROC           95% CI       F1   Spearman      ECE
Entropy only (B1)           0.5338 [0.483-0.584]   0.6657     0.0586   0.1039
+ Information Gain          0.3482 [0.297-0.396]   0.6143    -0.2629   0.2440
+ KL divergence             0.6974 [0.650-0.743]   0.6800     0.3419   0.1742
+ Confidence drop           0.3396 [0.292-0.389]   0.6200    -0.2777   0.2834
+ Semantic entropy          0.3110 [0.265-0.357]   0.6163    -0.3274   0.2963
Full composite              0.6405 [0.593-0.687]   0.6738     0.2433   0.1124

Note: SelfCheckGPT baseline will be added in 03_baselines.ipynb


In [ ]:
# Retune composite weights based on what actually works
# KL divergence is clearly the strongest signal (0.697)
# Entropy only is positive (0.534)
# The others are hurting — we need to flip their signs correctly

def compute_composite_score_v2(metrics):
    """
    Retuned composite based on empirical results.
    KL divergence is the dominant signal.
    """
    score = (
        metrics['kl_divergence']    * 0.50 +   # strongest signal
        metrics['entropy_only']     * 0.20 +   # positive signal
        -metrics['information_gain']* 0.15 +   # flip: low IG = hallucination
        -metrics['confidence_drop'] * 0.10 +   # flip: negative drop = hallucination
        -metrics['semantic_entropy']* 0.05     # flip
    )
    return score

# Recompute composite on full results
df_full['composite_v2'] = df_full.apply(
    lambda row: compute_composite_score_v2(row), axis=1
)

# Evaluate new composite
scores_v2 = df_full['composite_v2'].values
auroc_v2, ci_low, ci_high = compute_auroc_with_ci(scores_v2, labels)
f1_v2 = compute_f1(scores_v2, labels)
rho_v2, _ = spearmanr(scores_v2, labels)
ece_v2 = compute_ece(scores_v2, labels)

print("Retuned Composite Results:")
print(f"  AUROC:    {auroc_v2:.4f} [{ci_low:.3f}-{ci_high:.3f}]")
print(f"  F1:       {f1_v2:.4f}")
print(f"  Spearman: {rho_v2:.4f}")
print(f"  ECE:      {ece_v2:.4f}")
print()

# Also try KL divergence alone as upper bound reference
scores_kl = df_full['kl_divergence'].values
auroc_kl, _, _ = compute_auroc_with_ci(scores_kl, labels)
print(f"KL divergence alone AUROC: {auroc_kl:.4f}")
print(f"Improvement over KL alone: {auroc_v2 - auroc_kl:+.4f}")

# Save updated results
df_full['composite'] = df_full['composite_v2']
df_full.to_csv('results/ragtruth_results.csv', index=False)
print("\nUpdated results saved.")

Retuned Composite Results:
  AUROC:    0.6986 [0.652-0.741]
  F1:       0.6836
  Spearman: 0.3439
  ECE:      0.1687

KL divergence alone AUROC: 0.6974
Improvement over KL alone: +0.0012

Updated results saved.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split into train/test
feature_cols = ['entropy_only', 'information_gain', 
                'kl_divergence', 'confidence_drop', 'semantic_entropy']

X = df_full[feature_cols].values
y = labels

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Fit logistic regression composite
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)

# Get composite scores on test set
composite_lr = lr.predict_proba(X_test_scaled)[:, 1]

# Evaluate
auroc_lr, ci_low, ci_high = compute_auroc_with_ci(composite_lr, y_test)
f1_lr = compute_f1(composite_lr, y_test)
rho_lr, _ = spearmanr(composite_lr, y_test)
ece_lr = compute_ece(composite_lr, y_test)

print("Learned Composite (Logistic Regression) Results:")
print(f"  AUROC:    {auroc_lr:.4f} [{ci_low:.3f}-{ci_high:.3f}]")
print(f"  F1:       {f1_lr:.4f}")
print(f"  Spearman: {rho_lr:.4f}")
print(f"  ECE:      {ece_lr:.4f}")
print()
print("Feature weights (importance):")
for feat, coef in zip(feature_cols, lr.coef_[0]):
    print(f"  {feat:<25}: {coef:+.4f}")

# Save test results
df_test_results = df_full.iloc[
    list(range(len(df_full)))
].copy()
df_test_results.to_csv('results/ragtruth_results.csv', index=False)
print("\nSaved.")

Learned Composite (Logistic Regression) Results:
  AUROC:    0.7118 [0.628-0.794]
  F1:       0.6977
  Spearman: 0.3669
  ECE:      0.1073

Feature weights (importance):
  entropy_only             : +0.3097
  information_gain         : -0.0768
  kl_divergence            : +0.3865
  confidence_drop          : +0.4087
  semantic_entropy         : -0.7938

Saved.


In [ ]:
# Drop semantic entropy and retrain
feature_cols_v3 = ['entropy_only', 'information_gain', 
                   'kl_divergence', 'confidence_drop']

X_v3 = df_full[feature_cols_v3].values
X_train_v3, X_test_v3, y_train_v3, y_test_v3 = train_test_split(
    X_v3, y, test_size=0.3, random_state=42, stratify=y
)

scaler_v3 = StandardScaler()
X_train_v3_scaled = scaler_v3.fit_transform(X_train_v3)
X_test_v3_scaled = scaler_v3.transform(X_test_v3)

lr_v3 = LogisticRegression(random_state=42, max_iter=1000, C=0.5)
lr_v3.fit(X_train_v3_scaled, y_train_v3)

composite_v3 = lr_v3.predict_proba(X_test_v3_scaled)[:, 1]

auroc_v3, ci_low, ci_high = compute_auroc_with_ci(composite_v3, y_test_v3)
f1_v3 = compute_f1(composite_v3, y_test_v3)
rho_v3, _ = spearmanr(composite_v3, y_test_v3)
ece_v3 = compute_ece(composite_v3, y_test_v3)

print("Refined Composite (without semantic entropy):")
print(f"  AUROC:    {auroc_v3:.4f} [{ci_low:.3f}-{ci_high:.3f}]")
print(f"  F1:       {f1_v3:.4f}")
print(f"  Spearman: {rho_v3:.4f}")
print(f"  ECE:      {ece_v3:.4f}")
print()
print("Feature weights:")
for feat, coef in zip(feature_cols_v3, lr_v3.coef_[0]):
    print(f"  {feat:<25}: {coef:+.4f}")

# Also add entropy ratio as a new feature
df_full['entropy_ratio'] = df_full['entropy_only'] / (
    df_full['semantic_entropy'] + 1e-10
)

feature_cols_v4 = ['entropy_only', 'information_gain',
                   'kl_divergence', 'confidence_drop', 'entropy_ratio']

X_v4 = df_full[feature_cols_v4].values
X_train_v4, X_test_v4, y_train_v4, y_test_v4 = train_test_split(
    X_v4, y, test_size=0.3, random_state=42, stratify=y
)

scaler_v4 = StandardScaler()
X_train_v4_scaled = scaler_v4.fit_transform(X_train_v4)
X_test_v4_scaled = scaler_v4.transform(X_test_v4)

lr_v4 = LogisticRegression(random_state=42, max_iter=1000, C=0.5)
lr_v4.fit(X_train_v4_scaled, y_train_v4)

composite_v4 = lr_v4.predict_proba(X_test_v4_scaled)[:, 1]
auroc_v4, ci_low_v4, ci_high_v4 = compute_auroc_with_ci(composite_v4, y_test_v4)

print(f"\nWith entropy ratio feature added:")
print(f"  AUROC: {auroc_v4:.4f} [{ci_low_v4:.3f}-{ci_high_v4:.3f}]")

# Pick best model
if auroc_v4 > auroc_v3:
    print("\nBest composite: v4 (with entropy ratio)")
    best_auroc = auroc_v4
    best_ci = (ci_low_v4, ci_high_v4)
else:
    print("\nBest composite: v3 (without semantic entropy)")
    best_auroc = auroc_v3
    best_ci = (ci_low, ci_high)

print(f"Final composite AUROC: {best_auroc:.4f} [{best_ci[0]:.3f}-{best_ci[1]:.3f}]")

Refined Composite (without semantic entropy):
  AUROC:    0.7243 [0.643-0.798]
  F1:       0.7000
  Spearman: 0.3884
  ECE:      0.1238

Feature weights:
  entropy_only             : +0.0146
  information_gain         : +0.0133
  kl_divergence            : +0.5859
  confidence_drop          : -0.1381

With entropy ratio feature added:
  AUROC: 0.7159 [0.627-0.794]

Best composite: v3 (without semantic entropy)
Final composite AUROC: 0.7243 [0.643-0.798]


In [ ]:
# Add interaction features between strongest metrics
df_full['kl_x_entropy'] = df_full['kl_divergence'] * df_full['entropy_only']
df_full['kl_x_ig'] = df_full['kl_divergence'] * (-df_full['information_gain'])

feature_cols_v5 = ['entropy_only', 'information_gain',
                   'kl_divergence', 'confidence_drop',
                   'kl_x_entropy', 'kl_x_ig']

X_v5 = df_full[feature_cols_v5].values
X_train_v5, X_test_v5, y_train_v5, y_test_v5 = train_test_split(
    X_v5, y, test_size=0.3, random_state=42, stratify=y
)

scaler_v5 = StandardScaler()
X_train_v5_scaled = scaler_v5.fit_transform(X_train_v5)
X_test_v5_scaled = scaler_v5.transform(X_test_v5)

lr_v5 = LogisticRegression(random_state=42, max_iter=1000, C=0.5)
lr_v5.fit(X_train_v5_scaled, y_train_v5)

composite_v5 = lr_v5.predict_proba(X_test_v5_scaled)[:, 1]
auroc_v5, ci_low, ci_high = compute_auroc_with_ci(composite_v5, y_test_v5)
f1_v5 = compute_f1(composite_v5, y_test_v5)
rho_v5, _ = spearmanr(composite_v5, y_test_v5)
ece_v5 = compute_ece(composite_v5, y_test_v5)

print("Composite v5 (with interaction features):")
print(f"  AUROC:    {auroc_v5:.4f} [{ci_low:.3f}-{ci_high:.3f}]")
print(f"  F1:       {f1_v5:.4f}")
print(f"  Spearman: {rho_v5:.4f}")
print(f"  ECE:      {ece_v5:.4f}")
print()
print("Feature weights:")
for feat, coef in zip(feature_cols_v5, lr_v5.coef_[0]):
    print(f"  {feat:<25}: {coef:+.4f}")

# Save best scaler and model for later use in demo
import pickle
if auroc_v5 > 0.7243:
    print("\nv5 is best — saving model")
    with open('results/best_scaler.pkl', 'wb') as f:
        pickle.dump(scaler_v5, f)
    with open('results/best_model.pkl', 'wb') as f:
        pickle.dump(lr_v5, f)
    with open('results/best_features.pkl', 'wb') as f:
        pickle.dump(feature_cols_v5, f)
else:
    print("\nv3 remains best — saving that model")
    with open('results/best_scaler.pkl', 'wb') as f:
        pickle.dump(scaler_v3, f)
    with open('results/best_model.pkl', 'wb') as f:
        pickle.dump(lr_v3, f)
    with open('results/best_features.pkl', 'wb') as f:
        pickle.dump(feature_cols_v3, f)
print("Model saved for demo use.")

Composite v5 (with interaction features):
  AUROC:    0.7227 [0.644-0.800]
  F1:       0.6977
  Spearman: 0.3857
  ECE:      0.1103

Feature weights:
  entropy_only             : -0.2467
  information_gain         : -0.2540
  kl_divergence            : +0.2050
  confidence_drop          : -0.0456
  kl_x_entropy             : +0.4655
  kl_x_ig                  : -0.3295

v3 remains best — saving that model
Model saved for demo use.
